In [1]:
import numpy as np #imported to handle converting trigger to numpy
import matplotlib.pyplot as plt #imported only to set global plot parameters, matplotlib is imported natively in the relevant package files
import scipy #imported to handle loading a .mat file into python

import cfad #just stores the paths for everything and the treatment group information

from analysisUtils.probelocalisation import run_localisation #the actual pipeline function


# --- 1. GLOBAL CONFIGURATION ---
PARAMS = {
    # Hardware Specs
    'fs_raw': 30000,          # Acquisition Rate (Hz)
    'fs_lfp': 1000,           # Downsampled Rate (Hz)
    'voltage_scaling': 0.195, # uV per bit scaling, the .dat file saved as 16-bit binary needs to be scaled to voltages
    'probe_tip_offset': 720,  # Microns from tip with no channels

    # Analysis Windows
    'window_lfp': (-0.1, 0.5),    # Time window around trigger to average data around(s)
    'window_visual': (0.075, 0.150),# Window to find sSC sink (75 - 150ms post-stim)
    'bin_size_um': 15,            # Spatial bin for virtual probe, I think this is redundant now but still written to be safe, func does it dynmaically i believe
    
    # Bad Channel Thresholds
    'bad_ch_thresh': {
        'rms_low': 1.0,        # Dead (<1uV)
        'rms_high_factor': 15, # Noisy (>15x median)
        'corr_min': 0.6        # Artifact (<0.6 correlation to brain)
    },
    
    # Spike Statistics
    'spike_win_pre': (-0.2, 0.0), # Baseline window (s)
    'spike_win_post': (0, 0.2)# Visual response window (s)
}

#set global formatting parameters for plots, local function calls can still override this if needed
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'axes.linewidth': 1.5,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,  # Ensures fonts are editable in Adobe Illustrator
    'ps.fonttype': 42,
    'axes.spines.top': False,   # "Despining" for a cleaner look
    'axes.spines.right': False
})
Group = cfad.group #import treatment group information


Z:\Saij\scripts\analysisUtils\pipeline.py:315: SyntaxWarning: invalid escape sequence '\m'
  ax6.set_ylabel('Depth ($\mu m$)')
Z:\Saij\scripts\analysisUtils\visualisation\funcs.py:188: SyntaxWarning: invalid escape sequence '\p'
  color='gray', alpha=0.3, label='$\pm$ 1 SEM')


In [2]:

if __name__ == "__main__": #this line can be ignored, will be edited out eventually just here for legacy reasons (but don't delete yet)

    mice = [2] #list of mice to analyse
    for i in mice: #iterates over mice
        print("Analysing Mouse " + str(i))

        #define all the paths for the mouse, load triggers and find treatment group. cfad.data just stores path info as a dict for each mouse
        #this is ugly i know, i was an idiot when i first handled the pathing and now i'm in too deep to fix it 
        FILE_PATH = cfad.data['Neural']['Renewal']['Concatenated Data']['Mouse ' + str(i)] + r"/mouse" + str(i) + r"_renewal_concatenated_neural_data.dat"
        KS_DIR = cfad.data['Neural']['Renewal']['Concatenated Data']['kilosort4']['Mouse ' + str(i)]
        checker_triggers = np.array(scipy.io.loadmat(cfad.data['Neural']['Renewal']['Triggers']['Checkerboard']['Mouse ' + str(i)])["evt"]).flatten()
        OUTPUT_FOLDER = r'Z:\Saij\Data\Checkerboard\ScMouse' + str(i) #the paths here and below need to be editied out/obfuscated similar to cfad, just haven't gotten to it yet
        final_output_dir = r"Z:\Saij\ccf_channels"
        TRIGGERS = checker_triggers[1:]
        checkerboard_results_csv_path = r"Z:\Saij\Data\Checkerboard\nVEP.csv"
        probe_path = r"Z:\Saij\Herbs probes\Mouse " + str(i) + r"\probe_ccf_vox.npy"
        treatment_group = Group.get(i, "Unknown")

        #runs pipeline
        run_localisation(i, probe_path, FILE_PATH, KS_DIR, TRIGGERS, OUTPUT_FOLDER, final_output_dir, checkerboard_results_csv_path, PARAMS, False, treatment_group)


Analysing Mouse 2
--- PROCESSING: mouse2_renewal_concatenated_neural_data.dat ---
Step 1: Loading Probe Geometry...
Step 2: Calibrating (Detecting Bad Channels)...
  > Flagged 5 Bad Channels
Step 3: Extracting Diagnostic Sample...
Mean already computed, loading in as memmap
Step 5: Creating Virtual Probe (Average)...
Saved LFP binary to Z:\Saij\Data\Checkerboard\ScMouse2\data\mean_virtualLFP\mean_virtualLFP.dat
Saved LFP metadata to Z:\Saij\Data\Checkerboard\ScMouse2\data\mean_virtualLFP\mean_virtualLFP.json
Step 6: Calculating CSD...
  > sSC Center: 3060.0 um
Step 7: Integrating Spikes...
Generating Standalone Plots...
Saved: Mouse2_1_Raw_Continuous
Saved: Mouse2_2_Virtual_Continuous
Saved: Mouse2_3_Avg_LFP
Saved: Mouse2_4_CSD
Saved: Mouse2_5_KDE_spikes
Calculating depth-resolved Spike Density MUA...
MUA Spiking map generated.
Saved: Mouse2_6_Spiking_MUA_Map
Saved: Mouse2_7_PSD
Saved: Mouse2_8_Waveforms
Saved: Mouse2_11_Waveforms_bar
Saved: Mouse2_9_Waveforms_virtual
2. Generating Cha